# Week 1 micro-lab — impressive score or leaked evidence?

The same customer can appear in several rows. You will compare a random **row-level split** with a **customer-level split** and decide which result supports a future-customer claim.

Estimated time: 8–12 minutes. No file submission is required; preserve one sentence explaining the result.

## 1. Predict before running

Which validation score will be worse, and why? Discuss for 60 seconds before running the next cells.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

rng = np.random.default_rng(7)
n_customers, rows_per_customer = 160, 4
customer_id = np.repeat([f'CUST-{i:03d}' for i in range(n_customers)], rows_per_customer)
customer_value = np.repeat(rng.normal(100, 28, n_customers), rows_per_customer)
visit_activity = rng.normal(0, 1, n_customers * rows_per_customer)
future_value = customer_value + 6 * visit_activity + rng.normal(0, 4, len(customer_id))

data = pd.DataFrame({
    'customer_id': customer_id,
    'visit_activity': visit_activity,
    'future_value': future_value,
})
print(data.shape)
data.head()

In [ ]:
features = ['customer_id', 'visit_activity']
target = 'future_value'

model = Pipeline([
    ('prepare', ColumnTransformer([
        ('customer', OneHotEncoder(handle_unknown='ignore'), ['customer_id']),
        ('numeric', StandardScaler(), ['visit_activity']),
    ])),
    ('model', Ridge(alpha=0.2)),
])


## 2. Random row-level split

Rows are split independently. The same customers can therefore appear on both sides.

In [ ]:
row_train, row_valid = train_test_split(data, test_size=0.25, random_state=42)
overlap = set(row_train.customer_id) & set(row_valid.customer_id)

model.fit(row_train[features], row_train[target])
row_pred = model.predict(row_valid[features])
row_mae = mean_absolute_error(row_valid[target], row_pred)

print(f'Customers appearing in both train and validation: {len(overlap)}')
print(f'Row-level validation MAE: ${row_mae:,.2f}')

## 3. Customer-level split

Now every customer belongs entirely to training or validation. This tests performance for customers the model has never seen.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, valid_idx = next(splitter.split(data, groups=data.customer_id))
customer_train, customer_valid = data.iloc[train_idx], data.iloc[valid_idx]
customer_overlap = set(customer_train.customer_id) & set(customer_valid.customer_id)

model.fit(customer_train[features], customer_train[target])
customer_pred = model.predict(customer_valid[features])
customer_mae = mean_absolute_error(customer_valid[target], customer_pred)

print(f'Customers appearing in both train and validation: {len(customer_overlap)}')
print(f'Customer-level validation MAE: ${customer_mae:,.2f}')
print(f'MAE increase when customers are truly unseen: ${customer_mae - row_mae:,.2f}')

## 4. Preserve the decision

Complete the sentence below. Be ready to explain it to a peer.

In [ ]:
explanation = (
    'The __________ split is more credible for predicting a truly new customer because __________.'
)
print(explanation)